# 02 — Prepare native-schema SFT data

## Goal

Validate, render, measure and publish execution-verified Qwen
tool trajectories without syntactically translating third-party
harness traces. The output is a private, versioned dataset with
`messages` and rendered `text` columns.

In [ ]:
import subprocess
import sys
from pathlib import Path

# The Git pins supply current Unsloth/Qwen3.8 support. Transformers, TRL and
# Datasets deliberately use the mutually compatible versions from the adjacent
# official Unsloth Qwen3.5 27B notebook. Do not replace these with branch-head
# SHAs without resolving package metadata together first.
GIT_REVISIONS = {
    "unsloth": "c87fe20e32aca9ceb2dc5059c2987738f32446e8",
    "unsloth_zoo": "5b239e574f03ab3077c17e49aeef3cacfe7cdd4e",
}

import torch

torch_version = torch.__version__.split("+", 1)[0]
torch_minor = ".".join(torch_version.split(".")[:2])
torchao_by_torch = {"2.8": "0.16.0", "2.9": "0.16.0", "2.10": "0.16.0", "2.11": "0.18.0"}
xformers_by_torch = {"2.8": "0.0.32.post2", "2.9": "0.0.33.post1", "2.10": "0.0.34", "2.11": "0.0.34"}
if torch_minor not in torchao_by_torch:
    raise RuntimeError(
        f"No reviewed Colab dependency set for torch {torch.__version__}. "
        f"Expected one of {sorted(torchao_by_torch)}; update the compatibility matrix first."
    )

COMPATIBILITY_PINS = {
    "transformers": "5.3.0",
    "trl": "0.22.2",
    "datasets": "4.3.0",
    "peft": "0.19.0",
    "torchao": torchao_by_torch[torch_minor],
    "xformers": xformers_by_torch[torch_minor],
}
INSTALLER_REVISION = "colab-v2"
pin_key = "-".join(value.replace(".", "") for value in COMPATIBILITY_PINS.values())
git_key = "-".join(value[:8] for value in GIT_REVISIONS.values())
INSTALL_KEY = f"{INSTALLER_REVISION}-torch{torch_minor}-{git_key}-{pin_key}"
INSTALL_MARKER = Path(f"/content/.qwen38_env_{INSTALL_KEY}")
PIP_LOG = Path("/content/qwen38_pip_install.log")
FORCE_INSTALL = False

def install_phase(name: str, packages: list[str], *, no_deps: bool = False) -> None:
    command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "--upgrade-strategy",
        "only-if-needed",
        "--no-cache-dir",
        "--log",
        str(PIP_LOG),
    ]
    if no_deps:
        command.append("--no-deps")
    command.extend(packages)
    print(f"\n=== install phase: {name} ===")
    print("\n".join(f"  {package}" for package in packages))
    result = subprocess.run(command, check=False)
    if result.returncode:
        log_tail = (
            "\n".join(PIP_LOG.read_text(errors="replace").splitlines()[-120:])
            if PIP_LOG.exists()
            else "[pip did not create its log file]"
        )
        print(f"\n--- tail of {PIP_LOG} ---\n{log_tail}")
        raise RuntimeError(
            f"Package installation failed during {name!r} with exit code {result.returncode}. "
            f"The detailed log is at {PIP_LOG}."
        )

if FORCE_INSTALL or not INSTALL_MARKER.exists():
    if PIP_LOG.exists():
        PIP_LOG.unlink()
    install_phase("packaging tools", ["pip", "setuptools==80.9.0", "wheel>=0.42.0"])
    install_phase("Qwen3.8 training stack", [
        f"unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git@{GIT_REVISIONS['unsloth_zoo']}",
        f"unsloth @ git+https://github.com/unslothai/unsloth.git@{GIT_REVISIONS['unsloth']}",
        f"torch=={torch_version}",
        f"torchao=={COMPATIBILITY_PINS['torchao']}",
        f"transformers=={COMPATIBILITY_PINS['transformers']}",
        f"trl=={COMPATIBILITY_PINS['trl']}",
        f"datasets=={COMPATIBILITY_PINS['datasets']}",
        f"peft=={COMPATIBILITY_PINS['peft']}",
        "accelerate",
        "bitsandbytes",
        "trackio",
        "huggingface_hub>=0.34.0,<2.0",
        "hf_transfer",
        "sentencepiece>=0.2.0",
        "protobuf",
        "pytest",
        "jmespath",
    ])
    install_phase(
        "PyTorch-matched xFormers wheel",
        [f"xformers=={COMPATIBILITY_PINS['xformers']}"],
        no_deps=True,
    )
    INSTALL_MARKER.write_text(INSTALL_KEY)
    print("Packages installed. Restart the Colab runtime, then rerun this notebook from the top.")
else:
    print(f"Pinned environment already installed: {INSTALL_KEY}")

In [ ]:
import gc
import json
import os
import platform
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import torch
from huggingface_hub import login, whoami

if "GIT_REVISIONS" not in globals():
    raise RuntimeError(
        "This runtime was restarted. Rerun the notebook from the first cell; "
        "the install marker will skip the expensive package installation."
    )
if "COMPATIBILITY_PINS" not in globals():
    raise RuntimeError("Missing compatibility pins; rerun the notebook from the first cell.")

try:
    from google.colab import userdata
except ImportError:
    userdata = None

if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab G4 GPU runtime before continuing.")

gpu = torch.cuda.get_device_properties(0)
gpu_total_gib = gpu.total_memory / 1024**3
# A vendor-labelled 96 GB card can be reported as about 89.4 GiB because
# PyTorch converts the byte count with a binary divisor. Keep the floor well
# above the roughly 44.7 GiB reported for a 48 GB card without rejecting G4.
MIN_G4_TOTAL_GIB = 85.0
print(
    f"GPU: {gpu.name} ({gpu_total_gib:.1f} GiB total), "
    f"capability={torch.cuda.get_device_capability(0)}"
)
if gpu_total_gib < MIN_G4_TOTAL_GIB:
    raise RuntimeError(
        "This suite expects the nominal 96 GB Colab G4 runtime. "
        f"PyTorch reports {gpu_total_gib:.1f} GiB total; expected at least "
        f"{MIN_G4_TOTAL_GIB:.0f} GiB. A value near 45 GiB usually indicates "
        "the 48 GB GPU variant."
    )

# Make rerunning a notebook safe after another heavyweight notebook or a
# failed generation. Unsloth/TorchDynamo can retain compiled module references
# even after a Python variable is overwritten, so clear them before loading a
# fresh model. This does not free memory held by another live Python object.
for _stale_name in ("model", "tokenizer", "processor"):
    globals().pop(_stale_name, None)
gc.collect()
torch.cuda.empty_cache()
try:
    torch._dynamo.reset()
except AttributeError:
    pass

hf_token = userdata.get("HF_TOKEN") if userdata is not None else os.getenv("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Add a write-capable HF_TOKEN to Colab Secrets before continuing.")
login(token=hf_token, add_to_git_credential=False)
HF_USERNAME = whoami()["name"]

def package_version(name: str) -> str:
    try:
        return version(name)
    except PackageNotFoundError:
        return "missing"

observed_pins = {name: package_version(name) for name in COMPATIBILITY_PINS}
pin_mismatches = {
    name: {"expected": expected, "observed": observed_pins[name]}
    for name, expected in COMPATIBILITY_PINS.items()
    if observed_pins[name] != expected
}
if pin_mismatches:
    raise RuntimeError(
        "The runtime does not match the reviewed compatibility set. "
        f"Rerun the install cell with FORCE_INSTALL=True: {pin_mismatches}"
    )

RUN_ROOT = Path("/content/qwen38_runs")
RUN_ROOT.mkdir(parents=True, exist_ok=True)
runtime_manifest = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": gpu.name,
    "gpu_total_gib": round(gpu_total_gib, 2),
    "packages": {
        name: package_version(name)
        for name in ["unsloth", "unsloth_zoo", "transformers", "trl", "peft", "datasets"]
    },
    "git_revisions": GIT_REVISIONS,
    "compatibility_pins": COMPATIBILITY_PINS,
}
(RUN_ROOT / "runtime_manifest.json").write_text(json.dumps(runtime_manifest, indent=2))
print(json.dumps(runtime_manifest, indent=2))
print(f"Authenticated as {HF_USERNAME}")

## Load only the tokenizer and define the target schema

In [ ]:
from collections import Counter
from datasets import Dataset, concatenate_datasets, load_dataset
import numpy as np
from transformers import AutoTokenizer

MODEL_ID = "unsloth/Qwen3.8-27B"
SOURCE_DATASET_IDS = []  # Native-schema datasets only.
OUTPUT_DATASET_ID = f"{HF_USERNAME}/qwen38-code-native-sft-v0"
DEMO_MODE = True
AUDIT_PUBLIC_SCHEMAS = False
PUBLIC_AUDIT_IDS = [
    "nvidia/Nemotron-SFT-SWE-v3",
    "nvidia/Open-SWE-Traces",
    "nvidia/OpenCodeReasoning",
    "nvidia/OpenCodeInstruct",
]

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": "List files below a repository-relative directory.",
            "parameters": {
                "type": "object",
                "properties": {"path": {"type": "string"}},
                "required": ["path"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read a UTF-8 repository file with bounded output.",
            "parameters": {
                "type": "object",
                "properties": {"path": {"type": "string"}},
                "required": ["path"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search",
            "description": "Search repository text using a regular expression.",
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string"}},
                "required": ["query"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "apply_patch",
            "description": "Apply a unified diff to files inside the repository.",
            "parameters": {
                "type": "object",
                "properties": {"patch": {"type": "string"}},
                "required": ["patch"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "run_tests",
            "description": "Run an allow-listed repository test profile.",
            "parameters": {
                "type": "object",
                "properties": {"profile": {"type": "string", "enum": ["unit"]}},
                "required": ["profile"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "shell",
            "description": "Run a restricted allow-listed command. It is disabled in the pilot.",
            "parameters": {
                "type": "object",
                "properties": {"command": {"type": "string"}},
                "required": ["command"],
                "additionalProperties": False,
            },
        },
    },
]

def _without_arrow_nulls(value):
    """Remove null struct fields inserted by a Datasets/Arrow round trip."""
    if isinstance(value, dict):
        cleaned = {}
        for key, item in value.items():
            normalized = _without_arrow_nulls(item)
            if normalized is not None:
                cleaned[key] = normalized
        return cleaned
    if isinstance(value, list):
        return [_without_arrow_nulls(item) for item in value]
    return value

def canonical_tool_schema(tools: list[dict]) -> str:
    """Return a stable semantic fingerprint while retaining tool order."""
    return json.dumps(
        _without_arrow_nulls(tools),
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
    )

TOOL_SCHEMA_JSON = canonical_tool_schema(TOOLS)

def rendered_tool_schema(rendered_prompt: str) -> str:
    """Extract and canonicalise JSON tool declarations from a Qwen prompt."""
    start_tag = "<tools>"
    end_tag = "</tools>"
    if start_tag not in rendered_prompt or end_tag not in rendered_prompt:
        raise ValueError("Rendered prompt does not contain a <tools> block.")
    payload = rendered_prompt.split(start_tag, 1)[1].split(end_tag, 1)[0]
    try:
        rendered_tools = [
            json.loads(line)
            for line in payload.splitlines()
            if line.strip()
        ]
    except json.JSONDecodeError as exc:
        raise ValueError("Rendered <tools> block is not newline-delimited JSON.") from exc
    return canonical_tool_schema(rendered_tools)

def canonical_to_qwen(messages: list[dict]) -> list[dict]:
    """Fold an initial developer message into system for the HF tokenizer.

    The adapter performs the same mapping in training and deployment. The
    official safetensor tokenizer currently accepts system/user/assistant/tool.
    """
    converted = []
    pending_system = []
    for stored_message in messages:
        message = _without_arrow_nulls(stored_message)
        role = message["role"]
        if role in {"system", "developer"} and not converted:
            pending_system.append(str(message.get("content", "")))
            continue
        if pending_system:
            converted.append({"role": "system", "content": "\n\n".join(pending_system)})
            pending_system = []
        converted.append(message)
    if pending_system:
        converted.append({"role": "system", "content": "\n\n".join(pending_system)})
    return converted

def render_chat(messages: list[dict], *, add_generation_prompt: bool) -> str:
    return tokenizer.apply_chat_template(
        canonical_to_qwen(messages),
        tools=TOOLS,
        tokenize=False,
        add_generation_prompt=add_generation_prompt,
        enable_thinking=True,
        reasoning_effort="medium",
        preserve_thinking=True,
    )

## Load native examples or the format-only fixture

In [ ]:
demo_rows = [
    {
        "id": "fixture/native-tool-001",
        "repo_family": "fixture-clamp",
        "tool_schema_version": "qwen38-six-tools-v1",
        "tool_schema_json": TOOL_SCHEMA_JSON,
        "tools": TOOLS,
        "messages": [
            {"role": "developer", "content": "Inspect, edit narrowly, and run tests."},
            {"role": "user", "content": "Fix clamp and add regression coverage."},
            {"role": "assistant", "reasoning_content": "I should inspect the implementation first.", "content": "", "tool_calls": [{"type": "function", "function": {"name": "read_file", "arguments": {"path": "src/clamp.py"}}}]},
            {"role": "tool", "name": "read_file", "content": "def clamp(value, lower, upper):\n    return min(lower, max(upper, value))\n"},
            {"role": "assistant", "reasoning_content": "The min/max order is reversed.", "content": "", "tool_calls": [{"type": "function", "function": {"name": "apply_patch", "arguments": {"patch": "--- a/src/clamp.py\n+++ b/src/clamp.py\n@@ -1,2 +1,2 @@\n def clamp(value, lower, upper):\n-    return min(lower, max(upper, value))\n+    return max(lower, min(upper, value))\n"}}}]},
            {"role": "tool", "name": "apply_patch", "content": "patch applied"},
            {"role": "assistant", "reasoning_content": "I should verify the change.", "content": "", "tool_calls": [{"type": "function", "function": {"name": "run_tests", "arguments": {"profile": "unit"}}}]},
            {"role": "tool", "name": "run_tests", "content": "exit=0\n3 passed"},
            {"role": "assistant", "reasoning_content": "", "content": "Fixed the bound ordering and verified all tests pass."},
        ],
        "verification": {"all_required_tests_pass": True},
    },
    {
        "id": "fixture/native-tool-002",
        "repo_family": "fixture-parser",
        "tool_schema_version": "qwen38-six-tools-v1",
        "tool_schema_json": TOOL_SCHEMA_JSON,
        "tools": TOOLS,
        "messages": [
            {"role": "developer", "content": "Inspect the failing path and verify the focused change."},
            {"role": "user", "content": "Handle an empty CSV field as an empty list."},
            {"role": "assistant", "reasoning_content": "I should inspect the parser branch first.", "content": "", "tool_calls": [{"type": "function", "function": {"name": "read_file", "arguments": {"path": "src/parser.py"}}}]},
            {"role": "tool", "name": "read_file", "content": "def parse_field(value):\n    return value.split(',')\n"},
            {"role": "assistant", "reasoning_content": "The empty string needs an explicit branch.", "content": "", "tool_calls": [{"type": "function", "function": {"name": "apply_patch", "arguments": {"patch": "--- a/src/parser.py\n+++ b/src/parser.py\n@@ -1,2 +1,2 @@\n def parse_field(value):\n-    return value.split(',')\n+    return [] if value == '' else value.split(',')\n"}}}]},
            {"role": "tool", "name": "apply_patch", "content": "patch applied"},
            {"role": "assistant", "reasoning_content": "I should run the regression tests.", "content": "", "tool_calls": [{"type": "function", "function": {"name": "run_tests", "arguments": {"profile": "unit"}}}]},
            {"role": "tool", "name": "run_tests", "content": "exit=0\n4 passed"},
            {"role": "assistant", "reasoning_content": "", "content": "Added the empty-field branch and verified the parser tests."},
        ],
        "verification": {"all_required_tests_pass": True},
    },
]

if DEMO_MODE:
    raw_dataset = Dataset.from_list(demo_rows)
else:
    if not SOURCE_DATASET_IDS:
        raise ValueError("Set SOURCE_DATASET_IDS to native-schema datasets.")
    parts = [load_dataset(dataset_id, split="train") for dataset_id in SOURCE_DATASET_IDS]
    raw_dataset = concatenate_datasets(parts) if len(parts) > 1 else parts[0]

print(raw_dataset)
print(raw_dataset[0])

## Validate roles, tool calls and outcomes

In [ ]:
allowed_roles = {"system", "developer", "user", "assistant", "tool"}
allowed_tools = {item["function"]["name"] for item in TOOLS}
tool_specs = {item["function"]["name"]: item["function"] for item in TOOLS}

def validate_row(row: dict) -> list[str]:
    errors = []
    if row.get("tool_schema_version") != "qwen38-six-tools-v1":
        errors.append("wrong or missing tool_schema_version")
    if row.get("tool_schema_json") != TOOL_SCHEMA_JSON:
        errors.append("wrong or missing canonical tool_schema_json")
    if canonical_tool_schema(row.get("tools") or []) != TOOL_SCHEMA_JSON:
        errors.append("row tools differ from the deployment tool surface")
    messages = row.get("messages")
    if not isinstance(messages, list) or not messages:
        return ["messages must be a non-empty list"]
    pending_tools = []
    saw_tool_call = False
    for index, stored_message in enumerate(messages):
        message = _without_arrow_nulls(stored_message)
        role = message.get("role")
        if role not in allowed_roles:
            errors.append(f"message {index}: unexpected role {role!r}")
        if role == "assistant":
            if pending_tools:
                errors.append(f"message {index}: assistant turn before tool responses {pending_tools!r}")
            for call in message.get("tool_calls") or []:
                function = call.get("function", call)
                name = function.get("name")
                arguments = _without_arrow_nulls(function.get("arguments", {}))
                if name not in allowed_tools:
                    errors.append(f"message {index}: unknown tool {name!r}")
                    continue
                if not isinstance(arguments, dict):
                    errors.append(f"message {index}: arguments must be a mapping")
                    continue
                parameters = tool_specs[name]["parameters"]
                required = set(parameters.get("required", []))
                properties = set(parameters.get("properties", {}))
                if not required.issubset(arguments):
                    errors.append(f"message {index}: {name} missing required arguments")
                if parameters.get("additionalProperties") is False and not set(arguments).issubset(properties):
                    errors.append(f"message {index}: {name} has unknown arguments")
                pending_tools.append(name)
                saw_tool_call = True
        elif role == "tool":
            if not pending_tools:
                errors.append(f"message {index}: tool response without a pending call")
            else:
                expected = pending_tools.pop(0)
                if message.get("name") != expected:
                    errors.append(f"message {index}: response name {message.get('name')!r}, expected {expected!r}")
        elif pending_tools:
            errors.append(f"message {index}: unresolved tool responses {pending_tools!r}")
    if pending_tools:
        errors.append(f"trajectory ends with unresolved tool calls {pending_tools!r}")
    if not saw_tool_call:
        errors.append("native agent trajectory has no tool call")
    if not row.get("verification", {}).get("all_required_tests_pass", False):
        errors.append("trajectory is not execution-verified")
    return errors

validation = [validate_row(row) for row in raw_dataset]
bad = [(index, errors) for index, errors in enumerate(validation) if errors]
if bad:
    raise ValueError(f"Invalid native trajectories (first 20): {bad[:20]}")
print(f"Validated {len(raw_dataset)} native trajectories")

## Render and measure before truncation

In [ ]:
def render_row(row: dict) -> dict:
    messages = [_without_arrow_nulls(message) for message in row["messages"]]
    text = render_chat(messages, add_generation_prompt=False)
    token_count = len(tokenizer(text=text, add_special_tokens=False)["input_ids"])
    return {
        "messages": messages,
        "text": text,
        "token_count": token_count,
        "tools": TOOLS,
        "tool_schema_version": "qwen38-six-tools-v1",
        "tool_schema_json": TOOL_SCHEMA_JSON,
    }

prepared = raw_dataset.map(render_row)
lengths = np.array(prepared["token_count"])
percentiles = {
    percentile: float(np.percentile(lengths, percentile))
    for percentile in [50, 90, 95, 99]
}
print({"rows": len(prepared), "tokens": int(lengths.sum()), "percentiles": percentiles, "max": int(lengths.max())})
print(prepared[0]["text"][:5000])
assert "<tool_call>" in prepared[0]["text"]
assert "<tool_response>" in prepared[0]["text"]

## Optional: audit public schemas without importing their actions

In [ ]:
if AUDIT_PUBLIC_SCHEMAS:
    audit_rows = []
    for dataset_id in PUBLIC_AUDIT_IDS:
        try:
            sample = load_dataset(dataset_id, split="train", streaming=True).take(100)
            rows = list(sample)
            columns = sorted({key for row in rows for key in row})
            has_messages = sum(isinstance(row.get("messages"), list) for row in rows)
            audit_rows.append({
                "dataset": dataset_id,
                "rows": len(rows),
                "columns": columns,
                "message_rows": has_messages,
                "planning_direct_survival": 0,
            })
        except Exception as exc:
            audit_rows.append({"dataset": dataset_id, "error": f"{type(exc).__name__}: {exc}"})
    print(json.dumps(audit_rows, indent=2))
else:
    print("Public schema audit skipped; no third-party tool actions are imported by this notebook.")

## Freeze repository-family splits and publish privately

In [ ]:
repo_families = sorted(set(prepared["repo_family"]))
if len(repo_families) < 2:
    raise ValueError(
        "At least two repository families are required to create disjoint train and validation splits."
    )
validation_family_count = max(1, round(len(repo_families) * 0.10))
validation_family_count = min(validation_family_count, len(repo_families) - 1)
ranked_families = sorted(
    repo_families,
    key=lambda family: hashlib.sha256(family.encode()).hexdigest(),
)
validation_families = set(ranked_families[:validation_family_count])

def split_name(repo_family: str) -> str:
    return "validation" if repo_family in validation_families else "train"

prepared = prepared.map(lambda row: {"split": split_name(row["repo_family"])})
split_counts = Counter(prepared["split"])
print(split_counts)

from datasets import DatasetDict
dataset_dict = DatasetDict({
    split: prepared.filter(lambda row, expected=split: row["split"] == expected)
    for split in ("train", "validation")
})
if not dataset_dict["train"] or not dataset_dict["validation"]:
    raise RuntimeError(f"Split construction produced an empty partition: {split_counts}")

PUSH_DATASET = False
if PUSH_DATASET:
    if DEMO_MODE:
        raise RuntimeError("Refusing to publish the synthetic format fixture as training data.")
    dataset_dict.push_to_hub(OUTPUT_DATASET_ID, private=True)
    print(f"Pushed {OUTPUT_DATASET_ID}")
else:
    print("Set PUSH_DATASET=True only after reviewing rendered text and source licences.")

## Checks and next step

Do not use the demo fixture for capability training. Proceed to
SFT only after 100–300 successful native-schema trajectories are
validated, the repository-family split is frozen, and the
private dataset revision is recorded.